<a href="https://colab.research.google.com/github/aniget/SoftUni-AI-Integrations-for-developers/blob/main/Fine-Tuning/create_your_own_youtube_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 YouTube Channel Dataset Builder
### Using the YouTube Data API v3

---

## What you will learn

In this notebook you will learn how to:
- Authenticate with the YouTube Data API v3
- Search for videos on any topic of your choice
- Extract structured metadata (titles, descriptions, statistics, channels)
- Clean and export the data as a CSV dataset ready for fine-tuning or analysis

## How to adapt this notebook

Every section marked with 🔧 **YOUR TASK** is where you are expected to change the code for your own topic or subdomain.
Sections marked with 📖 **EXPLANATION** walk you through what the code does and why.

---

## Prerequisites

Before running this notebook you need a **YouTube Data API v3 key**.

Steps to get one:
1. Go to https://console.cloud.google.com/
2. Create a new project (or select an existing one)
3. Navigate to **APIs & Services → Library**
4. Search for **YouTube Data API v3** and click **Enable**
5. Go to **APIs & Services → Credentials → Create Credentials → API Key**
6. Copy your API key — you will paste it in the cell below

> ⚠️ **Keep your API key private.** Never share it publicly or commit it to GitHub.
> The free quota is **10,000 units per day**. Each search call costs 100 units.

---
## Step 1 — Install dependencies

📖 **EXPLANATION**

We use the official Google API Python client library to interact with YouTube's API.
`pandas` is used for organising and exporting our data.
`isodate` helps us convert YouTube's video duration format (ISO 8601) into human-readable minutes and seconds.

In [1]:
# Install required libraries
# google-api-python-client  : official YouTube Data API client
# pandas                    : data manipulation and CSV export
# isodate                   : parse ISO 8601 duration strings from the API

!pip install google-api-python-client pandas isodate --quiet

print('✅ Libraries installed successfully')

✅ Libraries installed successfully


---
## Step 2 — Import libraries and configure your API key

📖 **EXPLANATION**

We import the libraries we just installed and define our API key.
The `build()` function creates a client object that handles all communication with the YouTube API.

🔧 **YOUR TASK:** Paste your API key in the `API_KEY` variable below.

In [3]:
from googleapiclient.discovery import build
from google.colab import userdata
import pandas as pd
import isodate
import time
import json

# ─────────────────────────────────────────────────────────
# 🔧 YOUR TASK: Paste your YouTube Data API v3 key here
# ─────────────────────────────────────────────────────────
API_KEY = userdata.get('YOUTUBE_DATA_API_KEY')

# Build the YouTube API client
# 'youtube'  : the service name
# 'v3'       : the API version
youtube = build("youtube", "v3", developerKey=API_KEY)

print('✅ YouTube API client created successfully')

✅ YouTube API client created successfully


---
## Step 3 — Define your search topic and parameters

📖 **EXPLANATION**

This is the central configuration cell. Here you define:
- **SEARCH_QUERY**: The topic you want to collect videos about
- **MAX_RESULTS**: How many videos to collect per API call (max 50 per call)
- **MAX_PAGES**: How many pages of results to paginate through
- **LANGUAGE**: Filter results to a specific language (optional)
- **ORDER**: How to sort results — relevance, date, viewCount, rating

The YouTube API uses **pagination tokens** (nextPageToken) to navigate between pages of results.
Each page can return up to 50 results, so collecting 3 pages gives you up to 150 videos.

🔧 **YOUR TASK:** Change `SEARCH_QUERY` to your own topic of interest.
Examples: `'machine learning tutorials'`, `'medieval history'`, `'sourdough bread baking'`

In [19]:
# ─────────────────────────────────────────────────────────
# 🔧 YOUR TASK: Customise these parameters for your topic
# ─────────────────────────────────────────────────────────

SEARCH_QUERY = "gut microbiome and thoughts"   # <-- change this to your topic
MAX_RESULTS  = 30                           # results per page (max 50)
MAX_PAGES    = 5                            # how many pages to fetch
LANGUAGE     = "en"                         # language code, e.g. 'en', 'fr', 'de' (or None for any)
ORDER        = "relevance"                  # relevance | date | viewCount | rating
VIDEO_TYPE   = "video"                      # video | channel | playlist

# ─────────────────────────────────────────────────────────
# 🔧 OPTIONAL: Filter by video duration
# Options: 'any' | 'short' (<4 min) | 'medium' (4-20 min) | 'long' (>20 min)
# ─────────────────────────────────────────────────────────
VIDEO_DURATION = "medium"

print(f"🔍 Will search for: '{SEARCH_QUERY}'")
print(f"📄 Fetching up to {MAX_RESULTS * MAX_PAGES} results across {MAX_PAGES} page(s)")
print(f"🌐 Language filter: {LANGUAGE}")
print(f"📊 Sorted by: {ORDER}")

🔍 Will search for: 'gut microbiome and thoughts'
📄 Fetching up to 150 results across 5 page(s)
🌐 Language filter: en
📊 Sorted by: relevance


---
## Step 4 — Search for videos

📖 **EXPLANATION**

The `search().list()` endpoint is the main way to find videos on YouTube.
We pass our query and parameters, and it returns a list of video IDs and basic snippet data.

**Important:** The search endpoint only returns basic info (title, description, channel name, published date).
To get detailed statistics (views, likes, duration) we need a second API call in Step 5.

The `nextPageToken` is a cursor that points to the next page of results.
We loop until we reach `MAX_PAGES` or there are no more results.

In [20]:
def search_videos(query, max_results, max_pages, language, order, video_type, video_duration):
    """
    Search YouTube for videos matching the query.

    Returns a list of dicts with basic video info and the video_id
    needed for the detailed statistics call in Step 5.
    """
    all_items = []
    next_page_token = None

    for page_num in range(1, max_pages + 1):
        print(f"  Fetching page {page_num} of {max_pages}...")

        # Build the API request
        # 'part' specifies which data sections to return
        # 'snippet' contains title, description, channel, published date
        request = youtube.search().list(
            part           = "snippet",
            q              = query,
            type           = video_type,
            maxResults     = max_results,
            order          = order,
            relevanceLanguage = language,
            videoDuration  = video_duration,
            pageToken      = next_page_token   # None on the first call
        )

        # Execute the request and parse the response
        response = request.execute()

        # Extract the items from this page
        for item in response.get("items", []):
            # Guard: some search results may be channels or playlists, skip them
            if item["id"].get("kind") != "youtube#video":
                continue

            snippet = item["snippet"]

            all_items.append({
                "video_id"        : item["id"]["videoId"],
                "title"           : snippet.get("title", ""),
                "channel_title"   : snippet.get("channelTitle", ""),
                "channel_id"      : snippet.get("channelId", ""),
                "published_at"    : snippet.get("publishedAt", ""),
                "description"     : snippet.get("description", "")
                # "thumbnail_url"   : snippet.get("thumbnails", {}).get("high", {}).get("url", "")
            })

        # Check if there is a next page
        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            print("  ✅ No more pages available.")
            break

        # Be polite to the API: wait briefly between pages
        time.sleep(0.5)

    return all_items


print(f"🔍 Searching YouTube for: '{SEARCH_QUERY}'")
raw_results = search_videos(
    query          = SEARCH_QUERY,
    max_results    = MAX_RESULTS,
    max_pages      = MAX_PAGES,
    language       = LANGUAGE,
    order          = ORDER,
    video_type     = VIDEO_TYPE,
    video_duration = VIDEO_DURATION
)

print(f"\n📦 Retrieved {len(raw_results)} videos from search")
print("\nFirst result preview:")
print(json.dumps(raw_results[0], indent=2) if raw_results else 'No results found')

🔍 Searching YouTube for: 'gut microbiome and thoughts'
  Fetching page 1 of 5...
  Fetching page 2 of 5...
  ✅ No more pages available.

📦 Retrieved 55 videos from search

First result preview:
{
  "video_id": "B9RruLkAUm8",
  "title": "Your Gut Microbiome: The Most Important Organ You\u2019ve Never Heard Of | Erika Ebbel Angle | TEDxFargo",
  "channel_title": "TEDx Talks",
  "channel_id": "UCsT0YIqwnpJCM-mx7-gSA4Q",
  "published_at": "2019-12-12T15:21:52Z",
  "description": "NOTE FROM TED: Please do not look to this talk for medical advice. While some viewers might find advice provided in this talk to ...",
  "thumbnail_url": "https://i.ytimg.com/vi/B9RruLkAUm8/hqdefault.jpg"
}


---
## Step 5 — Enrich with detailed video statistics

📖 **EXPLANATION**

The search endpoint gives us titles and descriptions, but not view counts, likes, or durations.
For that we use the `videos().list()` endpoint with `part='statistics,contentDetails'`.

**API efficiency tip:** This endpoint accepts up to **50 video IDs in one call** as a comma-separated string.
We batch our video IDs in groups of 50 to minimise the number of API calls (and quota usage).

**Duration format:** YouTube returns duration in ISO 8601 format, e.g. `PT4M13S` means 4 minutes 13 seconds.
We use the `isodate` library to parse this into total seconds.

In [21]:
def get_video_statistics(video_ids):
    """
    Fetch detailed statistics and content details for a list of video IDs.

    Batches requests in groups of 50 (API limit) to minimise quota usage.
    Returns a dict mapping video_id -> stats dict.
    """
    stats_map = {}
    batch_size = 50  # YouTube API maximum IDs per request

    # Split the full list into batches of 50
    for i in range(0, len(video_ids), batch_size):
        batch = video_ids[i : i + batch_size]
        ids_string = ",".join(batch)  # API expects comma-separated IDs

        request = youtube.videos().list(
            part = "statistics,contentDetails",
            id   = ids_string
        )
        response = request.execute()

        for item in response.get("items", []):
            vid_id = item["id"]
            stats  = item.get("statistics", {})
            details = item.get("contentDetails", {})

            # Parse duration from ISO 8601 to total seconds
            duration_str = details.get("duration", "PT0S")
            try:
                duration_seconds = int(isodate.parse_duration(duration_str).total_seconds())
            except Exception:
                duration_seconds = 0

            stats_map[vid_id] = {
                "view_count"    : int(stats.get("viewCount", 0)),
                "like_count"    : int(stats.get("likeCount", 0)),
                "comment_count" : int(stats.get("commentCount", 0)),
                "duration_sec"  : duration_seconds,
                "duration_min"  : round(duration_seconds / 60, 1),
                "definition"    : details.get("definition", ""),  # 'hd' or 'sd'
                "caption"       : details.get("caption", "false")  # has captions?
            }

        time.sleep(0.3)  # avoid hitting rate limits

    return stats_map


# Collect all video IDs from our search results
video_ids = [item["video_id"] for item in raw_results]

print(f"📊 Fetching statistics for {len(video_ids)} videos...")
stats_map = get_video_statistics(video_ids)
print(f"✅ Statistics retrieved for {len(stats_map)} videos")

# Preview one entry
if stats_map:
    first_id = list(stats_map.keys())[0]
    print(f"\nSample stats for video {first_id}:")
    print(json.dumps(stats_map[first_id], indent=2))

📊 Fetching statistics for 55 videos...
✅ Statistics retrieved for 49 videos

Sample stats for video B9RruLkAUm8:
{
  "view_count": 2372117,
  "like_count": 54819,
  "comment_count": 1673,
  "duration_sec": 689,
  "duration_min": 11.5,
  "definition": "hd",
  "caption": "false"
}


---
## Step 6 — Merge search results and statistics into a DataFrame

📖 **EXPLANATION**

Now we combine the two data sources:
- `raw_results` → titles, descriptions, channel names (from search)
- `stats_map`   → views, likes, duration (from videos endpoint)

We also construct the full YouTube video URL from each video ID.

🔧 **YOUR TASK:** You can add or remove columns here depending on what your dataset needs.
For example, for an NLP dataset you might only want `title` and `description`.
For a recommendation system you might want `view_count`, `like_count`, and `duration_min`.

In [31]:
def build_dataframe(raw_results, stats_map):
    """
    Merge search results with video statistics into a single pandas DataFrame.
    """
    rows = []

    for item in raw_results:
        vid_id = item["video_id"]
        stats  = stats_map.get(vid_id, {})  # empty dict if stats are missing

        rows.append({
            # ── Identifiers ──────────────────────────────
            "video_id"      : vid_id,
            "url"           : f"https://www.youtube.com/watch?v={vid_id}",

            # ── Content metadata ─────────────────────────
            "title"         : item["title"],
            "description"   : item["description"],
            "published_at"  : item["published_at"],
            # "thumbnail_url" : item["thumbnail_url"],

            # ── Channel info ──────────────────────────────
            "channel_title" : item["channel_title"],
            "channel_id"    : item["channel_id"],

            # ── Statistics ───────────────────────────────
            "view_count"    : stats.get("view_count", 0),
            "like_count"    : stats.get("like_count", 0),
            "comment_count" : stats.get("comment_count", 0),

            # ── Content details ──────────────────────────
            "duration_sec"  : stats.get("duration_sec", 0),
            "duration_min"  : stats.get("duration_min", 0),
            "definition"    : stats.get("definition", ""),
            "has_captions"  : stats.get("caption", "false") == "true"
        })

    df = pd.DataFrame(rows)

    # Convert published_at to a proper datetime column
    df["published_at"] = pd.to_datetime(df["published_at"])

    return df


df = build_dataframe(raw_results, stats_map)

print(f"✅ DataFrame built: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)

✅ DataFrame built: 55 rows × 14 columns


,video_id,url,title,description,published_at,channel_title,channel_id,view_count,like_count,comment_count,duration_sec,duration_min,definition,has_captions
0,B9RruLkAUm8,https://www.youtube.com/watch?v=B9RruLkAUm8,Your Gut Microbiome: The Most Important Organ ...,NOTE FROM TED: Please do not look to this talk...,2019-12-12 15:21:52+00:00,TEDx Talks,UCsT0YIqwnpJCM-mx7-gSA4Q,2372117,54819,1673,689,11.5,hd,False
1,5h3Y4iNcN8g,https://www.youtube.com/watch?v=5h3Y4iNcN8g,How Your Gut Bacteria Controls Your Mood,UNLOCK YOUR BRAIN'S FULL POTENTIAL! My free 2-...,2021-06-02 12:45:00+00:00,Dr. Tracey Marks,UCL2QpphEeZFYwk6-WXD6hpA,2149901,102903,4518,479,8.0,hd,True
2,UjGMiChiUFc,https://www.youtube.com/watch?v=UjGMiChiUFc,Do Gut Microbes Control Your Personality? | Ka...,Biologist Kathleen McAuliffe dives into new re...,2024-01-22 19:27:12+00:00,TED,UCAuUUnT6oDeKwE6v1NGQxug,561543,18254,728,612,10.2,hd,True


---
## Step 7 — Explore and filter the dataset

📖 **EXPLANATION**

Before exporting, it is good practice to explore the data and apply filters.
Raw API results often contain very short videos, videos without captions, or duplicates.

🔧 **YOUR TASK:** Adjust the filters below based on your dataset needs.
For a fine-tuning dataset you might want only videos with captions and a minimum length.
For a trend analysis dataset you might filter by view count or date range.

In [23]:
# ── Basic statistics ─────────────────────────────────────
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total videos collected  : {len(df)}")
print(f"Unique channels         : {df['channel_title'].nunique()}")
print(f"Average views           : {df['view_count'].mean():,.0f}")
print(f"Average duration        : {df['duration_min'].mean():.1f} minutes")
print(f"Videos with captions    : {df['has_captions'].sum()} ({df['has_captions'].mean()*100:.0f}%)")
print(f"Date range              : {df['published_at'].min().date()} → {df['published_at'].max().date()}")

print("\n" + "=" * 50)
print("TOP 5 MOST VIEWED VIDEOS")
print("=" * 50)
print(df.nlargest(5, 'view_count')[['title', 'channel_title', 'view_count', 'duration_min']].to_string(index=False))

DATASET OVERVIEW
Total videos collected  : 55
Unique channels         : 39
Average views           : 670,798
Average duration        : 9.8 minutes
Videos with captions    : 21 (38%)
Date range              : 2013-11-05 → 2026-03-06

TOP 5 MOST VIEWED VIDEOS
                                                                                              title              channel_title  view_count  duration_min
                                                  How Bacteria Rule Over Your Body – The Microbiome Kurzgesagt – In a Nutshell    10582158           7.6
                                             How the food you eat affects your gut - Shilpa Ravella                     TED-Ed     6899414           5.2
 Food for thought: How your belly controls your brain | Ruairi Robertson | TEDxFulbrightSantaMonica                 TEDx Talks     5683296          14.5
Your Gut Microbiome: The Most Important Organ You’ve Never Heard Of | Erika Ebbel Angle | TEDxFargo                 TEDx Talks    

In [30]:
# ─────────────────────────────────────────────────────────
# 🔧 YOUR TASK: Set your own filter criteria
# ─────────────────────────────────────────────────────────

MIN_DURATION_MIN = 2     # minimum video length in minutes
MAX_DURATION_MIN = 60    # maximum video length in minutes
MIN_VIEW_COUNT   = 500   # ignore videos with very few views
REQUIRE_CAPTIONS = False # set to True if you need transcripts later

# Apply filters
df_filtered = df[
    (df["duration_min"] >= MIN_DURATION_MIN) &
    (df["duration_min"] <= MAX_DURATION_MIN) &
    (df["view_count"]   >= MIN_VIEW_COUNT)
]

if REQUIRE_CAPTIONS:
    df_filtered = df_filtered[df_filtered["has_captions"] == True]

# Drop exact duplicate titles (sometimes the same video appears in multiple pages)
df_filtered = df_filtered.drop_duplicates(subset=["video_id"])

print(f"Before filtering : {len(df)} videos")
print(f"After filtering  : {len(df_filtered)} videos")
df_filtered.head()

Before filtering : 55 videos
After filtering  : 47 videos


,video_id,url,title,description,published_at,thumbnail_url,channel_title,channel_id,view_count,like_count,comment_count,duration_sec,duration_min,definition,has_captions
0,B9RruLkAUm8,https://www.youtube.com/watch?v=B9RruLkAUm8,Your Gut Microbiome: The Most Important Organ ...,NOTE FROM TED: Please do not look to this talk...,2019-12-12 15:21:52+00:00,https://i.ytimg.com/vi/B9RruLkAUm8/hqdefault.jpg,TEDx Talks,UCsT0YIqwnpJCM-mx7-gSA4Q,2372117,54819,1673,689,11.5,hd,False
1,5h3Y4iNcN8g,https://www.youtube.com/watch?v=5h3Y4iNcN8g,How Your Gut Bacteria Controls Your Mood,UNLOCK YOUR BRAIN'S FULL POTENTIAL! My free 2-...,2021-06-02 12:45:00+00:00,https://i.ytimg.com/vi/5h3Y4iNcN8g/hqdefault.jpg,Dr. Tracey Marks,UCL2QpphEeZFYwk6-WXD6hpA,2149901,102903,4518,479,8.0,hd,True
2,UjGMiChiUFc,https://www.youtube.com/watch?v=UjGMiChiUFc,Do Gut Microbes Control Your Personality? | Ka...,Biologist Kathleen McAuliffe dives into new re...,2024-01-22 19:27:12+00:00,https://i.ytimg.com/vi/UjGMiChiUFc/hqdefault.jpg,TED,UCAuUUnT6oDeKwE6v1NGQxug,561543,18254,728,612,10.2,hd,True
3,VzPD009qTN4,https://www.youtube.com/watch?v=VzPD009qTN4,How Bacteria Rule Over Your Body – The Microbiome,What happens when microbes talk to your brain?...,2017-10-05 12:39:18+00:00,https://i.ytimg.com/vi/VzPD009qTN4/hqdefault.jpg,Kurzgesagt – In a Nutshell,UCsXVk37bltHxD1rDPwtNM8Q,10582158,269147,11546,458,7.6,hd,True
4,1sISguPDlhY,https://www.youtube.com/watch?v=1sISguPDlhY,How the food you eat affects your gut - Shilpa...,View full lesson: http://ed.ted.com/lessons/ho...,2017-03-23 15:07:29+00:00,https://i.ytimg.com/vi/1sISguPDlhY/hqdefault.jpg,TED-Ed,UCsooa4yRKGN_zEE8iknghZA,6899414,131941,2199,310,5.2,hd,True


---
## Step 8 — Export the dataset to CSV

📖 **EXPLANATION**

We export the cleaned DataFrame to a CSV file.
In Google Colab the file is saved to `/content/` and can be downloaded via the Files panel on the left.

🔧 **YOUR TASK:** Change the filename to reflect your topic.

In [33]:
# ─────────────────────────────────────────────────────────
# 🔧 YOUR TASK: Change the filename to match your topic
# ─────────────────────────────────────────────────────────
OUTPUT_FILENAME = SEARCH_QUERY + ".csv"  # <-- change this if you want a unique file name not based on the search query

output_path = f"/content/{OUTPUT_FILENAME}"
df_filtered.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Dataset saved to: {output_path}")
print(f"   Rows    : {len(df_filtered)}")
print(f"   Columns : {list(df_filtered.columns)}")
print("\n👉 Download it from the Files panel on the left sidebar in Colab.")

✅ Dataset saved to: /content/gut microbiome and thoughts.csv
   Rows    : 47
   Columns : ['video_id', 'url', 'title', 'description', 'published_at', 'thumbnail_url', 'channel_title', 'channel_id', 'view_count', 'like_count', 'comment_count', 'duration_sec', 'duration_min', 'definition', 'has_captions']

👉 Download it from the Files panel on the left sidebar in Colab.


---
## 🏁 Summary

You have now built a complete YouTube dataset pipeline:

| Step | What happened |
|------|---------------|
| 1 | Installed dependencies |
| 2 | Authenticated with the YouTube Data API |
| 3 | Defined a search topic and parameters |
| 4 | Fetched video search results (with pagination) |
| 5 | Enriched results with statistics and duration |
| 6 | Merged everything into a structured DataFrame |
| 7 | Explored and filtered the data |
| 8 | Exported to CSV |

---

## 💡 Ideas for extending this notebook

- **Add transcripts** using `youtube-transcript-api` and generate Q&A pairs with Claude
- **Channel deep-dive**: Given a channel ID, fetch all videos from that specific channel
- **Trend analysis**: Track how view counts change over time for a topic
- **Multi-language datasets**: Change `LANGUAGE` and collect videos in different languages
- **Comment scraping**: Use `commentThreads().list()` to collect audience reactions

---

## Step 9 — Build the CLI app (see Part 2 of this notebook)

In the next section we convert this notebook logic into a standalone Python CLI app
that your students can run from the terminal on any machine.

---
# Part 2 — Python CLI Application

📖 **EXPLANATION**

A CLI (Command Line Interface) app takes arguments from the terminal instead of from notebook cells.
This makes the tool reusable without opening a notebook every time.

We use Python's built-in `argparse` module which:
- Parses arguments passed when running the script (e.g. `--query`, `--max-results`)
- Automatically generates `--help` documentation
- Validates argument types (e.g. ensures `--max-results` is an integer)

The cell below writes the complete CLI app to a `.py` file.

🔧 **YOUR TASK:** After running the cell, try running the app with different topics:
```
python youtube_scraper.py --query "your topic here" --max-results 50
```

In [26]:
cli_script = '''
#!/usr/bin/env python3
"""
youtube_scraper.py
==================
A CLI tool to scrape YouTube video metadata by topic using the YouTube Data API v3.

Usage examples:
  python youtube_scraper.py --query "pear orchard" --api-key YOUR_KEY
  python youtube_scraper.py --query "machine learning" --max-results 50 --pages 3 --order viewCount
  python youtube_scraper.py --help
"""

import argparse
import time
import sys
import isodate
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


# ─────────────────────────────────────────────────────────
# Argument parser
# ─────────────────────────────────────────────────────────

def parse_args():
    parser = argparse.ArgumentParser(
        description="Scrape YouTube video metadata by topic and export to CSV.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )

    # Required arguments
    parser.add_argument("--query",      required=True,  help="Topic to search for, e.g. \'pear orchard\'")
    parser.add_argument("--api-key",    required=True,  help="Your YouTube Data API v3 key")

    # Optional arguments with defaults
    parser.add_argument("--max-results", type=int,   default=50,          help="Results per page (max 50)")
    parser.add_argument("--pages",       type=int,   default=2,           help="Number of pages to fetch")
    parser.add_argument("--order",       type=str,   default="relevance", choices=["relevance","date","viewCount","rating"], help="Sort order")
    parser.add_argument("--language",    type=str,   default="en",        help="Relevance language code (e.g. en, fr, de)")
    parser.add_argument("--duration",    type=str,   default="any",       choices=["any","short","medium","long"], help="Video duration filter")
    parser.add_argument("--min-views",   type=int,   default=0,           help="Minimum view count filter")
    parser.add_argument("--output",      type=str,   default="dataset.csv", help="Output CSV filename")
    parser.add_argument("--captions",    action="store_true",             help="Only include videos with captions")

    return parser.parse_args()


# ─────────────────────────────────────────────────────────
# Core functions (same logic as the notebook)
# ─────────────────────────────────────────────────────────

def search_videos(youtube, args):
    """Search YouTube and return a list of basic video info dicts."""
    all_items = []
    next_page_token = None

    for page in range(1, args.pages + 1):
        print(f"  [Page {page}/{args.pages}] Searching...")
        try:
            response = youtube.search().list(
                part              = "snippet",
                q                 = args.query,
                type              = "video",
                maxResults        = args.max_results,
                order             = args.order,
                relevanceLanguage = args.language,
                videoDuration     = args.duration,
                pageToken         = next_page_token
            ).execute()
        except HttpError as e:
            print(f"\n❌ API Error: {e}")
            sys.exit(1)

        for item in response.get("items", []):
            if item["id"].get("kind") != "youtube#video":
                continue
            s = item["snippet"]
            all_items.append({
                "video_id"      : item["id"]["videoId"],
                "title"         : s.get("title", ""),
                "channel_title" : s.get("channelTitle", ""),
                "channel_id"    : s.get("channelId", ""),
                "published_at"  : s.get("publishedAt", ""),
                "description"   : s.get("description", ""),
                "thumbnail_url" : s.get("thumbnails", {}).get("high", {}).get("url", "")
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break
        time.sleep(0.5)

    return all_items


def get_statistics(youtube, video_ids):
    """Batch fetch statistics and content details for a list of video IDs."""
    stats_map = {}
    for i in range(0, len(video_ids), 50):
        batch = ",".join(video_ids[i:i+50])
        try:
            response = youtube.videos().list(
                part = "statistics,contentDetails",
                id   = batch
            ).execute()
        except HttpError as e:
            print(f"Warning: could not fetch stats for batch — {e}")
            continue

        for item in response.get("items", []):
            stats   = item.get("statistics", {})
            details = item.get("contentDetails", {})
            try:
                dur_sec = int(isodate.parse_duration(details.get("duration", "PT0S")).total_seconds())
            except Exception:
                dur_sec = 0
            stats_map[item["id"]] = {
                "view_count"    : int(stats.get("viewCount", 0)),
                "like_count"    : int(stats.get("likeCount", 0)),
                "comment_count" : int(stats.get("commentCount", 0)),
                "duration_sec"  : dur_sec,
                "duration_min"  : round(dur_sec / 60, 1),
                "has_captions"  : details.get("caption", "false") == "true"
            }
        time.sleep(0.3)
    return stats_map


def build_and_filter_df(items, stats_map, args):
    """Merge search results with stats and apply user-defined filters."""
    rows = []
    for item in items:
        vid = item["video_id"]
        s = stats_map.get(vid, {})
        rows.append({
            "video_id"      : vid,
            "url"           : f"https://www.youtube.com/watch?v={vid}",
            "title"         : item["title"],
            "description"   : item["description"],
            "published_at"  : item["published_at"],
            "channel_title" : item["channel_title"],
            "channel_id"    : item["channel_id"],
            "thumbnail_url" : item["thumbnail_url"],
            "view_count"    : s.get("view_count", 0),
            "like_count"    : s.get("like_count", 0),
            "comment_count" : s.get("comment_count", 0),
            "duration_sec"  : s.get("duration_sec", 0),
            "duration_min"  : s.get("duration_min", 0),
            "has_captions"  : s.get("has_captions", False)
        })

    df = pd.DataFrame(rows)
    df["published_at"] = pd.to_datetime(df["published_at"])

    # Apply filters
    df = df[df["view_count"] >= args.min_views]
    if args.captions:
        df = df[df["has_captions"] == True]
    df = df.drop_duplicates(subset=["video_id"])

    return df


# ─────────────────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────────────────

def main():
    args = parse_args()

    print("\n🎬 YouTube Dataset Builder CLI")
    print("=" * 40)
    print(f"Query    : {args.query}")
    print(f"Pages    : {args.pages} × {args.max_results} results")
    print(f"Order    : {args.order}")
    print(f"Language : {args.language}")
    print(f"Output   : {args.output}")
    print("=" * 40)

    youtube = build("youtube", "v3", developerKey=args.api_key)

    print("\n🔍 Searching...")
    items = search_videos(youtube, args)
    print(f"   Found {len(items)} videos")

    print("\n📊 Fetching statistics...")
    video_ids = [i["video_id"] for i in items]
    stats_map = get_statistics(youtube, video_ids)

    print("\n🧹 Filtering and building dataset...")
    df = build_and_filter_df(items, stats_map, args)
    print(f"   Final dataset: {len(df)} rows")

    df.to_csv(args.output, index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved to: {args.output}")
    print(f"   Columns  : {list(df.columns)}")

    print("\n📌 Top 3 most viewed:")
    top3 = df.nlargest(3, "view_count")[["title","view_count","duration_min"]]
    for _, row in top3.iterrows():
        print(f"   {row[\'view_count\']:>10,} views | {row[\'duration_min\']:>5} min | {row[\'title\'][:60]}")


if __name__ == "__main__":
    main()
'''

# Write the CLI script to a .py file
with open("/content/youtube_scraper.py", "w") as f:
    f.write(cli_script)

print("✅ CLI app written to: /content/youtube_scraper.py")
print("\n📖 Run it in a terminal like this:")
print("   python youtube_scraper.py --query \"pear orchard\" --api-key YOUR_KEY")
print("\n📖 See all options:")
print("   python youtube_scraper.py --help")

✅ CLI app written to: /content/youtube_scraper.py

📖 Run it in a terminal like this:
   python youtube_scraper.py --query "pear orchard" --api-key YOUR_KEY

📖 See all options:
   python youtube_scraper.py --help


In [27]:
# Run --help to see all CLI options
!python /content/youtube_scraper.py --help

  File "/content/youtube_scraper.py", line 73
    print(f"
          ^
SyntaxError: unterminated f-string literal (detected at line 73)


In [28]:
# ─────────────────────────────────────────────────────────
# 🔧 YOUR TASK: Run the CLI app with your own arguments
# Replace YOUR_API_KEY and change the query to your topic
# ─────────────────────────────────────────────────────────

!python /content/youtube_scraper.py \
    --query "pear orchard cultivation" \
    --api-key "YOUR_API_KEY_HERE" \
    --max-results 50 \
    --pages 2 \
    --order viewCount \
    --min-views 1000 \
    --output cli_output_dataset.csv

  File "/content/youtube_scraper.py", line 73
    print(f"
          ^
SyntaxError: unterminated f-string literal (detected at line 73)
